<a href="https://colab.research.google.com/github/pbhise9/proj10_colab/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX 704 Week 10 Project
In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [62]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

fatal: destination path 'SQuAD-explorer' already exists and is not an empty directory.


In [63]:
import json

In [64]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [65]:
type(train_data)

dict

In [66]:
list(train_data.keys())

['data', 'version']

In [67]:
type(train_data["data"])

list

In [68]:
len(train_data["data"])

442

In [69]:
type(train_data["data"][0])

dict

In [70]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [71]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [72]:
len(train_data["data"][0]["paragraphs"])

55

In [73]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [74]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [75]:
# YOUR CHANGES HERE

import json
import pandas as pd

# Load JSON file
with open("SQuAD-explorer/dataset/train-v1.1.json", "r") as f:
    squad_data = json.load(f)

rows = []

# Iterate through documents
for doc in squad_data["data"]:
    title = doc["title"]

    # Iterate through paragraphs with index
    for idx, para in enumerate(doc["paragraphs"]):
        context = para["context"]

        rows.append({
            "document_title": title,
            "paragraph_index": idx,  # zero-based
            "paragraph_context": context
        })

# Convert to DataFrame
df = pd.DataFrame(rows)

# Save as TSV
df.to_csv("parsed.tsv", sep="\t", index=False)

Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [76]:
# YOUR CHANGES HERE

import pandas as pd
import json
from sklearn.feature_extraction.text import HashingVectorizer

# Read parsed paragraphs from Part 2
parsed_df = pd.read_csv("parsed.tsv", sep="\t")

# Create a 1024-dimensional paragraph encoder
vectorizer = HashingVectorizer(
    n_features=1024,
    alternate_sign=False,
    norm='l2',
    stop_words='english'
)

# Transform paragraph text into vectors
paragraph_matrix = vectorizer.transform(parsed_df["paragraph_context"])

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [77]:
# YOUR CHANGES HERE

# Convert sparse matrix rows into JSON-encoded lists
paragraph_vectors = [
    json.dumps(row.toarray().ravel().tolist())
    for row in paragraph_matrix
]

# Build submission dataframe
paragraph_vectors_df = pd.DataFrame({
    "document_title": parsed_df["document_title"],
    "paragraph_index": parsed_df["paragraph_index"],
    "paragraph_vector_json": paragraph_vectors
})

# Save as gzipped TSV
paragraph_vectors_df.to_csv(
    "paragraph-vectors.tsv.gz",
    sep="\t",
    index=False
)

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [78]:
# YOUR CHANGES HERE

import pandas as pd
import json
from sklearn.feature_extraction.text import HashingVectorizer

# Read questions
questions_df = pd.read_csv("questions.tsv", sep="\t")

# Find the text column automatically (everything except question_id)
text_col = [c for c in questions_df.columns if c != "question_id"][0]

# Use the same vector design as Part 3
question_vectorizer = HashingVectorizer(
    n_features=1024,
    alternate_sign=False,
    norm="l2",
    stop_words="english"
)

# Encode question text
question_matrix = question_vectorizer.transform(questions_df[text_col].fillna(""))

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [79]:
# YOUR CHANGES HERE

question_vectors = [
    json.dumps(row.toarray().ravel().tolist())
    for row in question_matrix
]

question_vectors_df = pd.DataFrame({
    "question_id": questions_df["question_id"],
    "question_vector_json": question_vectors
})

question_vectors_df.to_csv(
    "question-vectors.tsv",
    sep="\t",
    index=False
)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [80]:
# YOUR CHANGES HERE

import pandas as pd
import json
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Load paragraph vectors
paragraph_vectors_df = pd.read_csv("paragraph-vectors.tsv.gz", sep="\t")

# Load question vectors
question_vectors_df = pd.read_csv("question-vectors.tsv", sep="\t")

# Convert JSON vector strings to numpy arrays
paragraph_vectors = np.vstack(
    paragraph_vectors_df["paragraph_vector_json"].apply(json.loads).values
)

question_vectors = np.vstack(
    question_vectors_df["question_vector_json"].apply(json.loads).values
)

# Fit nearest neighbors model using Euclidean distance
nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn.fit(paragraph_vectors)

# Find top 5 nearest paragraphs for each question
distances, indices = nn.kneighbors(question_vectors)

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [81]:
# YOUR CHANGES HERE

rows = []

for q_idx, question_id in enumerate(question_vectors_df["question_id"]):
    for rank, para_idx in enumerate(indices[q_idx], start=1):
        rows.append({
            "question_id": question_id,
            "question_rank": rank,
            "document_title": paragraph_vectors_df.iloc[para_idx]["document_title"],
            "paragraph_index": paragraph_vectors_df.iloc[para_idx]["paragraph_index"]
        })

matches_df = pd.DataFrame(rows)

matches_df.to_csv("question-matches.tsv", sep="\t", index=False)

In [82]:
print(matches_df.head(10))
print(matches_df.groupby("question_id").size().head())

   question_id  question_rank                                document_title  \
0            1              1                                   Cork_(city)   
1            1              2                               Southern_Europe   
2            1              3                Geography_of_the_United_States   
3            1              4  Russian_Soviet_Federative_Socialist_Republic   
4            1              5                                     Christian   
5            4              1            BeiDou_Navigation_Satellite_System   
6            4              2            BeiDou_Navigation_Satellite_System   
7            4              3            BeiDou_Navigation_Satellite_System   
8            4              4            BeiDou_Navigation_Satellite_System   
9            4              5                 Nintendo_Entertainment_System   

   paragraph_index  
0                8  
1                4  
2               11  
3                6  
4               21  
5   

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [83]:
import pandas as pd

# Load the matches from Part 5
matches_df = pd.read_csv("question-matches.tsv", sep="\t")

# Sort by question_id, then question_rank
matches_df = matches_df.sort_values(["question_id", "question_rank"]).reset_index(drop=True)

# First 5 question_ids in ascending order
first_five_question_ids = sorted(matches_df["question_id"].unique())[:5]

# Keep only those questions
first_five_matches = matches_df[matches_df["question_id"].isin(first_five_question_ids)].copy()

# Worst match = lowest-quality among top 5 = rank 5
worst_paragraphs_df = (
    first_five_matches[first_five_matches["question_rank"] == 4][
        ["question_id", "document_title", "paragraph_index"]
    ]
    .sort_values("question_id")
    .reset_index(drop=True)
)

worst_paragraphs_df

,question_id,document_title,paragraph_index
0,1,Russian_Soviet_Federative_Socialist_Republic,6
1,4,BeiDou_Navigation_Satellite_System,7
2,7,Molotov%E2%80%93Ribbentrop_Pact,30
3,10,Roman_Republic,43
4,13,Southeast_Asia,9


In [84]:
worst_paragraphs_df.to_csv("worst-paragraphs.tsv", sep="\t", index=False)

In [85]:
print(worst_paragraphs_df)
print(worst_paragraphs_df.shape)

   question_id                                document_title  paragraph_index
0            1  Russian_Soviet_Federative_Socialist_Republic                6
1            4            BeiDou_Navigation_Satellite_System                7
2            7               Molotov%E2%80%93Ribbentrop_Pact               30
3           10                                Roman_Republic               43
4           13                                Southeast_Asia                9
(5, 3)


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.